# 3.4 Evaluating Tree-Based Models

## Course 3: Advanced Classification Models for Student Success

## Introduction

In this notebook, we perform a **thorough evaluation** of our three tuned tree-based models. We go beyond simple accuracy or ROC-AUC, and examine precision-recall trade-offs, and confusion matrices, all critical for deploying models trained on imbalanced data in a higher education context.

### Learning Objectives

1. Generate and interpret Precision-Recall curves
2. Analyze confusion matrices for each model

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
project_path = '/content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3'
data_filepath = '/data/'
course3_models = '/models/'

### Data Setup

In [ ]:
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
from plotly.subplots import make_subplots

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ARTIFACT_DIR = f'{project_path}{course3_models}'
feature_columns = joblib.load(f'{ARTIFACT_DIR}feature_columns_assignment.pkl')
train_medians   = joblib.load(f'{ARTIFACT_DIR}train_medians_assignment.pkl')

# --- Load TEST data only ---
df_testing_assignment = pd.read_csv(f'{project_path}{data_filepath}testing_assignment.csv')
df_testing_assignment['DEPARTED'] = (df_testing_assignment['SEM_2_STATUS'] != 'E').astype(int)

numeric_features = ['HS_GPA','HS_MATH_GPA','HS_ENGL_GPA','UNITS_ATTEMPTED_1','UNITS_ATTEMPTED_2',
    'UNITS_COMPLETED_1','UNITS_COMPLETED_2','DFW_UNITS_1','DFW_UNITS_2','GPA_1','GPA_2',
    'DFW_RATE_1','DFW_RATE_2','GRADE_POINTS_1','GRADE_POINTS_2']
categorical_features = ['RACE_ETHNICITY','GENDER','FIRST_GEN_STATUS','COLLEGE']

test_enc = pd.get_dummies(df_testing_assignment[numeric_features + categorical_features],
                          columns=categorical_features, drop_first=True)

# Reconstruct the exact training feature set: add missing dummies as 0, drop unseen ones
test_enc = test_enc.reindex(columns=feature_columns, fill_value=0)
# Impute with TRAIN medians (column-aligned), never test's own
test_enc = test_enc.fillna(train_medians)

X_test, y_test = test_enc, df_testing_assignment['DEPARTED']

print(f"Test data prepared: {X_test.shape[0]:,} samples | {X_test.shape[1]} features")
print(f"Departure rate: {y_test.mean():.2%} (test)")

### Load Pre-Trained Models

In [ ]:
import joblib

# Dictionary to hold the loaded models
models = {}
# Dictionaries to store predictions and probabilities for evaluation
predictions = {}
probabilities = {}

# Define the filenames for the models to be loaded
model_filenames = {
    'Decision Tree': 'dt_tuned_f1_assignment.pkl',
    'Random Forest': 'rf_tuned_f1_assignment.pkl',
    'XGBoost': 'xgb_tuned_f1_assignment.pkl'
}

# Load each model, make predictions, and store probabilities
for name, filename in model_filenames.items():
    model_path = f'{project_path}{course3_models}{filename}'
    model = joblib.load(model_path)
    models[name] = model
    predictions[name] = model.predict(X_test)
    probabilities[name] = model.predict_proba(X_test)[:, 1]

print("Models loaded and predictions/probabilities generated.")
print(f"Models loaded: {list(models.keys())}")
print(f"Predictions keys: {list(predictions.keys())}")
print(f"Probabilities keys: {list(probabilities.keys())}")

## 2. Precision-Recall Curves

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

plt.figure(figsize=(10, 7))

# Calculate baseline prevalence
baseline_prevalence = y_test.sum() / len(y_test)

# Plot PR curves for each model
for name, prob in probabilities.items():
    precision, recall, _ = precision_recall_curve(y_test, prob)
    ap_score = average_precision_score(y_test, prob)
    plt.plot(recall, precision, label=f'{name} (AP = {ap_score:.2f})')

# Plot baseline
plt.plot([0, 1], [baseline_prevalence, baseline_prevalence], linestyle='--', color='gray', label='Baseline (Class Prevalence)')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve for Tree-Based Models')
plt.legend()
plt.grid(True)
plt.ylim([0.0, 1.05])
plt.xlim([0.0, 1.0])
plt.show()

## 3. Confusion Matrices

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Confusion Matrices for Tree-Based Models', fontsize=16)

for i, (name, y_pred) in enumerate(predictions.items()):
    ax = axes[i]
    disp = ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap='Blues')
    ax.set_title(f'{name} Confusion Matrix')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent suptitle overlap
plt.show()

## 4. Summary - Performance Metrics in the Context of Imbalanced Data

When working with imbalanced datasets, standard metrics like accuracy can be misleading. For instance, a model predicting the majority class for all instances might achieve high accuracy but be useless for the minority class, which is often the class of interest.

### Precision-Recall (PR) Curves

PR curves offer a more informative view of model performance on imbalanced data, especially when the positive class is the minority. Unlike ROC curves, which can paint an overly optimistic picture for highly imbalanced datasets, PR curves focus directly on the trade-off between Precision and Recall. This visualization helps us understand how a model performs across different thresholds, providing critical insight into its ability to correctly identify positive instances without generating too many false alarms.

### Confusion Matrices

Confusion matrices provide a detailed breakdown of correct and incorrect classifications. For imbalanced data, examining the raw counts of True Positives (TP), False Positives (FP), False Negatives (FN), and True Negatives (TN) is invaluable. This granular view helps identify where the model struggles, especially concerning the minority class. For example, a high number of False Negatives for the minority class (e.g., students who depart but are predicted to enroll) can be immediately identified, highlighting critical areas for model improvement and understanding the practical implications of prediction errors.

### Next Module

We will compare these tree-based models against Regularized Logistic Regression in a systematic model comparison framework.

**Proceed to:** `Module 4: Model Comparison and Selection`